# START

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
def single_violin(expr,gene):
    # Create a violin plot of expression values
    plt.figure(figsize=(8, 6))
    sns.violinplot(y=expr, color="skyblue", inner="box")
    # Calculate statistics
    mean_expression = expr.mean()
    median_expression = expr.median()


    # Add mean and median lines
    plt.axhline(mean_expression, color="red",  linewidth=1)
    plt.axhline(median_expression, color="green",  linewidth=1)
    plt.text(0, mean_expression, f"Mean: {mean_expression:.2f}", color="red", fontsize=8)
    plt.text(0, median_expression, f"Median: {median_expression:.2f}", color="green", fontsize=8)

    # Customize the plot
    plt.title(f"Distribution of Expression Values for {gene}")
    plt.ylabel("Expression Value")
    plt.xlabel("All Samples (ROIs)")
    plt.show()

# Load the spatial data

In [3]:
import pandas as pd
import numpy as np
import os
from LEAPdata import get_slide_id, get_leap_ids

In [4]:
OME_LEVEL = 2 # should be level 1 but haven't extracted at 20x yet
DOWNSAMPLE = 2**OME_LEVEL
segment='CD45+'

#file_path = 'C:/Users/hooll/Dropbox/19 PostGrad/00 CB Lab/01 projects/06 MultiModal/data/LEAP'
#input_path = 'E:/01-data/LEAP'
root_path = "C:/Users/hooll/Dropbox/19 PostGrad/00 CB Lab/01 projects/06 MultiModal/data"
rawdata_path = os.path.join(root_path,'01-raw')
outcome_path = os.path.join(root_path,'02-input','outcomes')

In [5]:

sample_metadata = pd.read_excel(os.path.join(rawdata_path,'metadata',"meta_data.xlsx"), index_col=0)
samples = pd.read_excel(os.path.join(rawdata_path,'metadata',"samples.xlsx"), index_col=0)

In [6]:
samples.head()

,Biobank_ID,Slide_ID,H&E,Response,ECT_Carbo
LEAP_ID,,,,,
LEAP001,19005858,1,YES,Non-Responder,NO
LEAP003,21002266,2,YES,Non-Responder,NO
LEAP005,22001707,3,YES,Non-Responder,NO
LEAP008,22001709,4,YES,Responder,YES
LEAP009,19005859,5,YES,Responder,NO


In [7]:
sample_metadata = sample_metadata.set_index('DCC_ID')
sample_metadata.head()

,Scan.Name,Panel,Roi,Segment,Aoi,Area,Tags,Nuclei,ROICoordinateX,ROICoordinateY,...,NegGeoMean,NegGeoSD,NegProbes,lib_size,countOfLowEprGene,percentOfLowEprGene,ruv_W1,ruv_W2,ruv_W3,ruv_W4
DCC_ID,,,,,,,,,,,,,,,,,,,,,
DSP-1001660008694-G-A02.dcc,"LEAP015, LEAP016",(v1.0) Human NGS Whole Transcriptome Atlas RNA,1.0,CD45+,CD45+-aoi-001,7186.20,"PanCK+, CD45+, CD45+",119,76238.40625,34222.16797,...,1.008594,1.080060,False,22680,0,0.000000,-0.023221,-0.007044,-0.017423,0.022888
DSP-1001660008694-G-A03.dcc,"LEAP015, LEAP016",(v1.0) Human NGS Whole Transcriptome Atlas RNA,1.0,PanCK+,PanCK+-aoi-001,12989.46,"PanCK+, CD45+, PanCK+",239,76238.40625,34222.16797,...,1.115328,1.329805,False,51774,13492,73.087757,0.008550,-0.022099,0.023774,0.020598
DSP-1001660008694-G-A05.dcc,"LEAP015, LEAP016",(v1.0) Human NGS Whole Transcriptome Atlas RNA,2.0,PanCK+,PanCK+-aoi-001,14576.49,"PanCK+, CD45+, PanCK+",246,75587.60938,31947.06250,...,1.405232,1.600153,False,77467,11795,63.894908,0.006329,-0.020188,0.015802,-0.001984
DSP-1001660008694-G-A06.dcc,"LEAP015, LEAP016",(v1.0) Human NGS Whole Transcriptome Atlas RNA,3.0,CD45+,CD45+-aoi-001,5066.89,"PanCK+, CD45+, CD45+",85,76252.50000,29973.66797,...,1.017262,1.114309,False,20049,0,0.000000,-0.035865,-0.006889,-0.027053,0.014727
DSP-1001660008694-G-A07.dcc,"LEAP015, LEAP016",(v1.0) Human NGS Whole Transcriptome Atlas RNA,3.0,PanCK+,PanCK+-aoi-001,41874.55,"PanCK+, CD45+, PanCK+",796,76252.50000,29973.66797,...,3.029957,1.887786,False,274462,12122,65.666306,0.007790,-0.016314,0.018326,0.004932


In [8]:
sample_metadata.columns

Index(['Scan.Name', 'Panel', 'Roi', 'Segment', 'Aoi', 'Area', 'Tags', 'Nuclei',
       'ROICoordinateX', 'ROICoordinateY', 'Scan.Date', 'Scan.Width',
       'Scan.Height', 'Scan.Offset.X', 'Scan.Offset.Y', 'WTA.Atlas',
       'X.v1.0..Human.NGS.TCR.Profiling.RNA.Add.On', 'Sample_ID', 'nuclei',
       'area', 'NTC', 'NucleiZero', 'NucleiArea', 'AreaReads',
       'NegGeoMean_Hs_R_NGS_WTA_v1.0', 'NegGeoSD_Hs_R_NGS_WTA_v1.0', 'LOQ',
       'GenesDetected', 'GeneDetectionRate', 'Tissue_ID', 'LEAP_ID',
       'Patient_ID', 'Sample_Type', 'Biobank_ID', 'Response', 'Date_Sectioned',
       'Format', 'FORCE', 'ECT_Carbo', 'NACT_Group', 'sTILs', 'sTILs_Status',
       'BRCA1_Status', 'BRCA2_Status', 'Age', 'Ethnicity', 'Year_Diagnosis',
       'DistRecTime', 'DthTime', 'DistRecTime_Cat', 'DthTime_Cat', 'N_Stage',
       'T_Stage', 'Grade', 'RCB_Scores', 'RCB_Group', 'Extreme_NR',
       'Epithelial_E', 'Region_R', 'sample_id', 'NegGeoMean', 'NegGeoSD',
       'NegProbes', 'lib_size', 'countOfLo

In [9]:
#raw_matrix = pd.read_csv(os.path.join(file_path,"raw_exprs.csv"), index_col=0)
#expression_matrix = raw_matrix.astype(float)

In [10]:
norm_matrix = pd.read_csv(os.path.join(rawdata_path,'sge',"norm_counts.csv"), 
                 sep=";",               # Semicolon separator
                 decimal=",",          # Comma as decimal point
                 quotechar='"',        # Handle quoted strings
                 index_col=0)          # Use the first column as row index (gene name)

norm_expr_matrix = norm_matrix.astype(float)

In [11]:
norm_expr_matrix.head()

,DSP-1001660008694-G-A02.dcc,DSP-1001660008694-G-A03.dcc,DSP-1001660008694-G-A05.dcc,DSP-1001660008694-G-A06.dcc,DSP-1001660008694-G-A07.dcc,DSP-1001660008694-G-A08.dcc,DSP-1001660008694-G-A09.dcc,DSP-1001660008694-G-A10.dcc,DSP-1001660008694-G-A11.dcc,DSP-1001660008694-G-A12.dcc,...,DSP-1001660024022-D-H03.dcc,DSP-1001660024022-D-H04.dcc,DSP-1001660024022-D-H05.dcc,DSP-1001660024022-D-H06.dcc,DSP-1001660024022-D-H07.dcc,DSP-1001660024022-D-H08.dcc,DSP-1001660024022-D-H09.dcc,DSP-1001660024022-D-H10.dcc,DSP-1001660024022-D-H11.dcc,DSP-1001660024022-D-H12.dcc
A2M,8.736860,7.686246,6.830261,7.730451,6.309591,8.198439,7.265407,4.789404,7.753964,8.135174,...,7.983239,6.874335,8.424866,7.618854,7.389477,7.106261,8.151385,7.365071,7.574850,8.325490
ACADM,5.502417,6.386417,7.157426,5.524405,5.528383,4.645522,6.229605,5.613796,5.595107,5.964793,...,5.669373,4.502646,5.404367,6.027267,5.447129,6.175177,5.866024,6.476642,5.846188,5.779037
ACADS,5.095812,5.979818,5.901526,5.018215,5.150049,5.039447,6.287200,4.991735,5.445171,6.055979,...,4.593009,6.173799,5.020409,5.541564,6.217577,5.050971,4.432642,4.116373,3.700987,5.415778
ACAT1,5.551031,7.379331,6.400583,5.686364,7.293440,4.725142,6.379487,4.794026,5.549913,6.974626,...,5.027639,6.449585,6.061562,4.335965,6.585729,5.871773,5.276001,5.155276,5.833425,5.592112
ACVRL1,4.744912,4.843521,6.156299,4.606600,5.220992,5.710566,4.878334,4.835873,3.998791,5.775298,...,4.578111,3.757643,3.932488,6.160220,4.338776,4.984209,4.143997,3.803191,4.515328,5.561657


In [12]:
raw_matrix = pd.read_csv(os.path.join(rawdata_path,'sge',"raw_counts.csv"), 
                 sep=";",               # Semicolon separator
                 decimal=",",          # Comma as decimal point
                 quotechar='"',        # Handle quoted strings
                 index_col=0)          # Use the first column as row index (gene name)

In [13]:

expression_matrix = raw_matrix.astype(float)

In [14]:
expression_matrix.head()

,DSP-1001660008694-G-A02.dcc,DSP-1001660008694-G-A03.dcc,DSP-1001660008694-G-A05.dcc,DSP-1001660008694-G-A06.dcc,DSP-1001660008694-G-A07.dcc,DSP-1001660008694-G-A08.dcc,DSP-1001660008694-G-A09.dcc,DSP-1001660008694-G-A10.dcc,DSP-1001660008694-G-A11.dcc,DSP-1001660008694-G-A12.dcc,...,DSP-1001660024022-D-H03.dcc,DSP-1001660024022-D-H04.dcc,DSP-1001660024022-D-H05.dcc,DSP-1001660024022-D-H06.dcc,DSP-1001660024022-D-H07.dcc,DSP-1001660024022-D-H08.dcc,DSP-1001660024022-D-H09.dcc,DSP-1001660024022-D-H10.dcc,DSP-1001660024022-D-H11.dcc,DSP-1001660024022-D-H12.dcc
A2M,5.0,11.0,12.0,2.0,26.0,12.0,25.0,1.0,53.0,7.0,...,21.0,12.0,314.0,20.0,7.0,52.0,134.0,19.0,59.0,458.0
ACADM,1.0,4.0,12.0,1.0,12.0,1.0,11.0,2.0,10.0,2.0,...,7.0,2.0,39.0,7.0,2.0,35.0,51.0,23.0,26.0,67.0
ACADS,1.0,2.0,3.0,1.0,6.0,1.0,8.0,1.0,6.0,2.0,...,4.0,7.0,28.0,5.0,4.0,20.0,22.0,5.0,5.0,44.0
ACAT1,1.0,9.0,7.0,1.0,45.0,1.0,11.0,1.0,9.0,4.0,...,7.0,9.0,55.0,2.0,5.0,35.0,57.0,15.0,34.0,47.0
ACVRL1,1.0,1.0,4.0,1.0,7.0,2.0,3.0,1.0,2.0,2.0,...,6.0,1.0,11.0,8.0,1.0,21.0,29.0,7.0,14.0,42.0


In [15]:
# Replace '.' with '-' in column names
expression_matrix.columns = expression_matrix.columns.str.replace('.', '-', regex=False)
expression_matrix.columns = expression_matrix.columns.str.replace('-dcc', '.dcc', regex=False)
expression_matrix.columns
# Ensure data is in a usable format
#print(expression_matrix.head())  # Check expression matrix
#expression_matrix.columns

Index(['DSP-1001660008694-G-A02.dcc', 'DSP-1001660008694-G-A03.dcc',
       'DSP-1001660008694-G-A05.dcc', 'DSP-1001660008694-G-A06.dcc',
       'DSP-1001660008694-G-A07.dcc', 'DSP-1001660008694-G-A08.dcc',
       'DSP-1001660008694-G-A09.dcc', 'DSP-1001660008694-G-A10.dcc',
       'DSP-1001660008694-G-A11.dcc', 'DSP-1001660008694-G-A12.dcc',
       ...
       'DSP-1001660024022-D-H03.dcc', 'DSP-1001660024022-D-H04.dcc',
       'DSP-1001660024022-D-H05.dcc', 'DSP-1001660024022-D-H06.dcc',
       'DSP-1001660024022-D-H07.dcc', 'DSP-1001660024022-D-H08.dcc',
       'DSP-1001660024022-D-H09.dcc', 'DSP-1001660024022-D-H10.dcc',
       'DSP-1001660024022-D-H11.dcc', 'DSP-1001660024022-D-H12.dcc'],
      dtype='object', length=5096)

In [16]:
expression_matrix.head()

,DSP-1001660008694-G-A02.dcc,DSP-1001660008694-G-A03.dcc,DSP-1001660008694-G-A05.dcc,DSP-1001660008694-G-A06.dcc,DSP-1001660008694-G-A07.dcc,DSP-1001660008694-G-A08.dcc,DSP-1001660008694-G-A09.dcc,DSP-1001660008694-G-A10.dcc,DSP-1001660008694-G-A11.dcc,DSP-1001660008694-G-A12.dcc,...,DSP-1001660024022-D-H03.dcc,DSP-1001660024022-D-H04.dcc,DSP-1001660024022-D-H05.dcc,DSP-1001660024022-D-H06.dcc,DSP-1001660024022-D-H07.dcc,DSP-1001660024022-D-H08.dcc,DSP-1001660024022-D-H09.dcc,DSP-1001660024022-D-H10.dcc,DSP-1001660024022-D-H11.dcc,DSP-1001660024022-D-H12.dcc
A2M,5.0,11.0,12.0,2.0,26.0,12.0,25.0,1.0,53.0,7.0,...,21.0,12.0,314.0,20.0,7.0,52.0,134.0,19.0,59.0,458.0
ACADM,1.0,4.0,12.0,1.0,12.0,1.0,11.0,2.0,10.0,2.0,...,7.0,2.0,39.0,7.0,2.0,35.0,51.0,23.0,26.0,67.0
ACADS,1.0,2.0,3.0,1.0,6.0,1.0,8.0,1.0,6.0,2.0,...,4.0,7.0,28.0,5.0,4.0,20.0,22.0,5.0,5.0,44.0
ACAT1,1.0,9.0,7.0,1.0,45.0,1.0,11.0,1.0,9.0,4.0,...,7.0,9.0,55.0,2.0,5.0,35.0,57.0,15.0,34.0,47.0
ACVRL1,1.0,1.0,4.0,1.0,7.0,2.0,3.0,1.0,2.0,2.0,...,6.0,1.0,11.0,8.0,1.0,21.0,29.0,7.0,14.0,42.0


## Only use data where we have both the gene expression and the metadata

keep only rows that are not pellets

In [17]:
sample_metadata = sample_metadata[sample_metadata['Sample_Type'] != 'PELLET']

In [18]:
rois_to_use = list(set(sample_metadata.index).intersection(set(expression_matrix.columns)))
len(rois_to_use)

4963

In [19]:
sample_metadata = sample_metadata.loc[rois_to_use]
expression_matrix = expression_matrix[rois_to_use]
norm_expr_matrix = norm_expr_matrix[rois_to_use]

In [20]:
sample_metadata.shape

(4963, 70)

# Identify hold out test set
need to group by patient


All segments from a patient must go into either train or test.

The test set must contain:

At least one CD45+ and one PanCK+ in Segment

At least one RESECTION and one DIAGNOSTIC in Sample_Type



In [21]:
patient_segment_counts = sample_metadata.groupby(['Patient_ID', 'Segment']).size().unstack(fill_value=0)
patient_segment_counts

Segment,CD45+,Full ROI,PanCK+,PanCK-,PanCK-/CD45-,TLS
Patient_ID,,,,,,
GST15102,2,0,2,0,3,0
GST17605,5,0,5,0,0,0
GST17723,7,0,7,0,5,0
GST18770,4,0,4,0,0,0
GST18928,5,0,5,0,0,0
...,...,...,...,...,...,...
GST48495,46,0,117,26,0,0
GST48581,23,0,21,0,0,0
GST48631,0,0,19,0,0,0


In [22]:
patient_segment_counts[(patient_segment_counts['CD45+'] == 0) & 
                                         (patient_segment_counts['PanCK+'] == 0)]

Segment,CD45+,Full ROI,PanCK+,PanCK-,PanCK-/CD45-,TLS
Patient_ID,,,,,,


In [23]:
patient_summary = sample_metadata.groupby('Patient_ID').agg({
    'Segment': lambda x: set(x),
    'Sample_Type': lambda x: set(x)
})
patient_summary

,Segment,Sample_Type
Patient_ID,,
GST15102,"{CD45+, PanCK+, PanCK-/CD45-}",{RESECTION}
GST17605,"{PanCK+, CD45+}",{RESECTION}
GST17723,"{PanCK-/CD45-, PanCK+, CD45+}",{RESECTION}
GST18770,"{PanCK+, CD45+}",{RESECTION}
GST18928,"{PanCK+, CD45+}",{RESECTION}
...,...,...
GST48495,"{PanCK-, PanCK+, CD45+}","{RESECTION, DIAGNOSTIC}"
GST48581,"{PanCK+, CD45+}",{DIAGNOSTIC}
GST48631,{PanCK+},"{RESECTION, DIAGNOSTIC}"


In [ ]:
# Step 1: Get unique Patient_IDs
unique_patients = sample_metadata['Patient_ID'].unique()

# Step 2: Randomly sample 10% of patients
np.random.seed(42)
sampled_patients = np.random.choice(unique_patients,
                                     size=int(0.1 * len(unique_patients)),
                                     replace=False)



In [ ]:
# Step 3: Get all rows for those patients
train_metadata = sample_metadata[ ~sample_metadata['Patient_ID'].isin(sampled_patients)] 
test_metadata = sample_metadata[sample_metadata['Patient_ID'].isin(sampled_patients)]

these patients were chosen randomly using the code above and used to generate the  data but the seed was not set correctly so the above code changes every time 

In [26]:
train_patients = {'GST15102',
 'GST17605',
 'GST17723',
 'GST18770',
 'GST18928',
 'GST18944',
 'GST19918',
 'GST23088',
 'GST23677',
 'GST24143',
 'GST25373',
 'GST25518',
 'GST26237',
 'GST26311',
 'GST26322',
 'GST26353',
 'GST26423',
 'GST26448',
 'GST26501',
 'GST26791',
 'GST26854',
 'GST26965',
 'GST27131',
 'GST27235',
 'GST27321',
 'GST27337',
 'GST27498',
 'GST27506',
 'GST27925',
 'GST29819',
 'GST30473',
 'GST35861',
 'GST36019',
 'GST36564',
 'GST36713',
 'GST36718',
 'GST36776',
 'GST36789',
 'GST37108',
 'GST37122',
 'GST37411',
 'GST37685',
 'GST37926',
 'GST41084',
 'GST41151',
 'GST43189',
 'GST43338',
 'GST43395',
 'GST43651',
 'GST43654',
 'GST43962',
 'GST44149',
 'GST44189',
 'GST44222',
 'GST44440',
 'GST44790',
 'GST44827',
 'GST46489',
 'GST46735',
 'GST46801',
 'GST47130',
 'GST47289',
 'GST47381',
 'GST47384',
 'GST47471',
 'GST47765',
 'GST47796',
 'GST47859',
 'GST47883',
 'GST48110',
 'GST48134',
 'GST48176',
 'GST48323',
 'GST48454',
 'GST48455',
 'GST48456',
 'GST48495',
 'GST48581',
 'GST48631',
 'GST48798',
 'GST49284'}
len(train_patients)

81

In [27]:
for p in train_patients:
    print(p)

GST47765
GST18770
GST36713
GST44440
GST47381
GST44790
GST48495
GST47130
GST26791
GST30473
GST27498
GST43651
GST47859
GST47883
GST17605
GST23677
GST48110
GST48323
GST27235
GST37685
GST15102
GST48798
GST49284
GST26448
GST48176
GST27337
GST27506
GST44222
GST44149
GST26322
GST26501
GST26353
GST43395
GST36789
GST18928
GST26311
GST36776
GST36019
GST47384
GST46489
GST48134
GST25518
GST26237
GST26854
GST41084
GST43338
GST48581
GST43189
GST18944
GST29819
GST37122
GST47289
GST46735
GST43962
GST46801
GST27925
GST48455
GST27321
GST36718
GST37108
GST48631
GST27131
GST26965
GST19918
GST48454
GST25373
GST44189
GST48456
GST37926
GST35861
GST44827
GST17723
GST41151
GST43654
GST36564
GST23088
GST47796
GST37411
GST26423
GST24143
GST47471


In [28]:
test_patients = {'GST23703',
 'GST25222',
 'GST26416',
 'GST26519',
 'GST26889',
 'GST43294',
 'GST47380',
 'GST47661'}
len(test_patients)

8

In [29]:
train_metadata = sample_metadata[sample_metadata['Patient_ID'].isin(train_patients)]
test_metadata = sample_metadata[sample_metadata['Patient_ID'].isin(test_patients)]

check we have totally separated test and train

In [30]:
(train_metadata.shape[0] + test_metadata.shape[0]) == sample_metadata.shape[0]

True

In [31]:
rois_train = list(set(train_metadata.index))
rois_test = list(set(test_metadata.index))

In [197]:
print(f"train rois: {len(rois_train)}")
print(f"test rois: {len(rois_test)}")
print(f"{len(rois_test)/len(rois_train):%}")

train rois: 4518
test rois: 445
9.849491%


In [32]:
(len(rois_test)+len(rois_train))==len(rois_to_use)

True

In [33]:
expression_train = expression_matrix[rois_train]
expression_test = expression_matrix[rois_test]

In [34]:
print("Original Sample_Type proportions:")
print(sample_metadata['Sample_Type'].value_counts(normalize=True))

print("\nSampled Sample_Type proportions:")
print(train_metadata['Sample_Type'].value_counts(normalize=True))
print("\nSampled Sample_Type proportions:")
print(test_metadata['Sample_Type'].value_counts(normalize=True))

Original Sample_Type proportions:
Sample_Type
DIAGNOSTIC    0.812009
RESECTION     0.158775
Pre-NACT      0.015515
1_1           0.010478
2_2           0.003224
Name: proportion, dtype: float64

Sampled Sample_Type proportions:
Sample_Type
DIAGNOSTIC    0.828021
RESECTION     0.139885
Pre-NACT      0.017043
1_1           0.011510
2_2           0.003541
Name: proportion, dtype: float64

Sampled Sample_Type proportions:
Sample_Type
DIAGNOSTIC    0.649438
RESECTION     0.350562
Name: proportion, dtype: float64


In [35]:
expression_test.shape

(18460, 445)

In [36]:
test_metadata.shape

(445, 70)

# Gene Expression

In [37]:
rois_cd45_train = list(train_metadata[train_metadata['Segment']=='CD45+'].index)
rois_panck_train = list(train_metadata[train_metadata['Segment']=='PanCK+'].index)

In [38]:
rois_cd45_test = list(test_metadata[test_metadata['Segment']=='CD45+'].index)
rois_panck_test = list(test_metadata[test_metadata['Segment']=='PanCK+'].index)

In [39]:
log_transformed_expr = expression_matrix.apply(np.log1p).T

In [40]:
sample_metadata = sample_metadata.sort_index()
#final_expression_matrix = expression_matrix.T.astype(float).sort_index()
#log_transformed_expr = final_expression_matrix.apply(np.log1p)
norm_expr_matrix = norm_expr_matrix.T.astype(float).sort_index()
norm_expr_matrix.head()

,A2M,ACADM,ACADS,ACAT1,ACVRL1,PSEN1,ADA,SGCA,ADRB2,ADRB3,...,TNFSF18,CCL23,WNT6,VEGFD,GLI1,NDUFA3,HAMP,ADH1B,CCL3,CST2
DSP-1001660008694-G-A02.dcc,8.736860,5.502417,5.095812,5.551031,4.744912,5.276005,5.327490,4.709154,4.748537,4.524060,...,4.857875,4.948848,5.321573,5.291184,4.677396,5.727129,4.605154,4.636586,5.203187,4.491745
DSP-1001660008694-G-A03.dcc,7.686246,6.386417,5.979818,7.379331,4.843521,6.654290,4.760808,5.580089,4.966828,4.663518,...,4.584349,5.137348,5.168634,4.940911,5.690619,6.918611,4.605043,4.866588,5.595081,4.665243
DSP-1001660008694-G-A05.dcc,6.830261,7.157426,5.901526,6.400583,6.156299,5.308238,6.423473,5.900474,4.529480,4.094271,...,3.912445,5.333651,5.264247,6.460309,5.582917,6.148312,4.851412,4.275327,6.059923,5.115517
DSP-1001660008694-G-A06.dcc,7.730451,5.524405,5.018215,5.686364,4.606600,5.247726,5.229229,4.408976,4.577911,4.159722,...,4.524535,4.641437,5.110279,5.151759,4.355345,5.772854,4.260822,4.277445,4.997191,4.287743
DSP-1001660008694-G-A07.dcc,6.309591,5.528383,5.150049,7.293440,5.220992,5.758353,5.327022,4.846551,3.283847,5.352596,...,3.451089,3.901585,4.276544,4.759025,4.926420,6.396389,4.545595,4.159937,5.266447,4.925050


## Remove Contamination

Tumour segments have contamination from plasma cells so we need to remove these genes before doing our analysis

make sure we are only doing this for PanCK+ AOIs

In [ ]:
tumour_aois = sorted(rois_panck_test + rois_panck_train)

len(tumour_aois)

2381

In [42]:
plasma_genes = ["IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHM", "IGHA1", "IGKC", "IGLL5"]

In [43]:
tumour_expr = norm_expr_matrix.loc[tumour_aois,:]
tumour_expr.head()
#tumour_expr.loc[plasma_genes,:].head()

,A2M,ACADM,ACADS,ACAT1,ACVRL1,PSEN1,ADA,SGCA,ADRB2,ADRB3,...,TNFSF18,CCL23,WNT6,VEGFD,GLI1,NDUFA3,HAMP,ADH1B,CCL3,CST2
DSP-1001660008694-G-A03.dcc,7.686246,6.386417,5.979818,7.379331,4.843521,6.654290,4.760808,5.580089,4.966828,4.663518,...,4.584349,5.137348,5.168634,4.940911,5.690619,6.918611,4.605043,4.866588,5.595081,4.665243
DSP-1001660008694-G-A05.dcc,6.830261,7.157426,5.901526,6.400583,6.156299,5.308238,6.423473,5.900474,4.529480,4.094271,...,3.912445,5.333651,5.264247,6.460309,5.582917,6.148312,4.851412,4.275327,6.059923,5.115517
DSP-1001660008694-G-A07.dcc,6.309591,5.528383,5.150049,7.293440,5.220992,5.758353,5.327022,4.846551,3.283847,5.352596,...,3.451089,3.901585,4.276544,4.759025,4.926420,6.396389,4.545595,4.159937,5.266447,4.925050
DSP-1001660008694-G-A09.dcc,7.265407,6.229605,6.287200,6.379487,4.878334,4.933541,5.575112,5.491822,6.108288,4.448252,...,3.826982,4.350966,5.731811,5.006641,4.079268,6.575408,5.051638,5.732952,5.391057,5.649282
DSP-1001660008694-G-A11.dcc,7.753964,5.595107,5.445171,5.549913,3.998791,4.752077,6.448923,4.563710,4.936684,5.173806,...,4.609001,4.857005,5.331441,5.662894,4.226408,6.211358,5.106878,4.272509,4.452907,5.884296


check correlation of all genes with known plasma genes to identify other genes that are likely from plasma cells rather than tumour epithelial 

In [44]:
# compute correlations of every gene vs. the mean plasma score
plasma_score = tumour_expr[plasma_genes].mean(axis=1)

corrs = tumour_expr.corrwith(plasma_score, axis=0)  # correlation of each gene with plasma_score
corrs = corrs.sort_values(ascending=False)

# turn into dataframe
corr_df = pd.DataFrame({
    "Gene": corrs.index,
    "Correlation_with_Plasma": corrs.values
})

Inspect correlation and remove anything over 0.7

In [45]:
corr_df[corr_df['Correlation_with_Plasma']>0.7]

,Gene,Correlation_with_Plasma
0,IGHG3,0.976263
1,IGHG1,0.974007
2,IGHG4,0.973143
3,IGHG2,0.973092
4,IGKC,0.969368
5,IGLL5,0.896842
6,IGHM,0.751077
7,IGHA1,0.726305


In [46]:
contamination_genes = corr_df.loc[corr_df['Correlation_with_Plasma']>0.7,'Gene'].to_list()

we should consider anything over 0.5 as possible plasma - JCHAIN and MZB1 are definite plasma genes

In [47]:
cond = (corr_df['Correlation_with_Plasma']<=0.7)&(corr_df['Correlation_with_Plasma']>0.5)
corr_df[cond]

,Gene,Correlation_with_Plasma
8,JCHAIN,0.601160
9,MZB1,0.585446


In [48]:
contamination_genes = contamination_genes + corr_df.loc[cond,'Gene'].to_list()
contamination_genes

['IGHG3',
 'IGHG1',
 'IGHG4',
 'IGHG2',
 'IGKC',
 'IGLL5',
 'IGHM',
 'IGHA1',
 'JCHAIN',
 'MZB1']

In [49]:
cond = (corr_df['Correlation_with_Plasma']<=0.5)&(corr_df['Correlation_with_Plasma']>0.4)
corr_df[cond]

,Gene,Correlation_with_Plasma
10,DERL3,0.476888
11,LYZ,0.476679
12,CD74,0.469337
13,CD79A,0.459737
14,TRBC1,0.456418
15,TENT5C,0.454135
16,C1QB,0.448888
17,PIM2,0.448880
18,C1QC,0.447108
19,HLA-DPA1,0.446821


Strongly immune-specific (should consider removing as we only want tumour-epithelial)

- CD79A (corr 0.46) → B-cell receptor, B lineage only.
- TRBC1 (corr 0.46) → T-cell receptor β chain, T lineage only.
- C1QA, C1QB, C1QC (corr 0.40, 0.45, 0.45) → complement genes, macrophage/myeloid lineage.
- ADA2 (corr 0.43) → myeloid/monocyte.
- LYZ (corr 0.48) → lysozyme, classic macrophage/neutrophil marker.
- IL10RA (corr 0.42) → monocyte/macrophage, not epithelial.
- EVI2B, CORO1A, WIPF1 (corr 0.42, 0.41, 0.41) → leukocyte cytoskeleton/trafficking, T/NK/myeloid associated.

In [50]:
final_norm_matrix = norm_expr_matrix.copy()

set expression to zero for contamination genes in tumour epithelial

In [51]:
final_norm_matrix.loc[tumour_aois,contamination_genes] = 0
# check we have only set to 0 on the PanCK+ AOIs
final_norm_matrix.loc[:,contamination_genes]

,IGHG3,IGHG1,IGHG4,IGHG2,IGKC,IGLL5,IGHM,IGHA1,JCHAIN,MZB1
DSP-1001660008694-G-A02.dcc,6.321806,6.486105,7.910064,6.526511,6.411107,5.731868,6.819770,5.011164,5.068021,5.381198
DSP-1001660008694-G-A03.dcc,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
DSP-1001660008694-G-A05.dcc,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
DSP-1001660008694-G-A06.dcc,7.993664,8.253587,8.136558,8.223948,9.065838,7.226121,6.803185,5.836918,5.588335,6.079026
DSP-1001660008694-G-A07.dcc,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
DSP-1001660024022-D-H08.dcc,12.405782,12.672189,13.227088,12.129353,12.467595,10.147085,7.915245,7.571767,5.479322,7.427696
DSP-1001660024022-D-H09.dcc,7.342027,7.469779,7.941272,7.368211,7.414136,4.386362,4.661535,3.905513,3.287045,4.385806
DSP-1001660024022-D-H10.dcc,7.130658,7.233049,7.577838,7.171591,7.478592,4.059951,4.348946,0.534741,2.254485,4.497559
DSP-1001660024022-D-H11.dcc,11.400145,11.588678,12.208536,11.288754,11.512960,8.785918,6.600135,6.661187,4.583878,6.710718


## Setup AnnData object

use the normalised, batch-corrected data to find the HVGs and SVGs

In [52]:
import scanpy as sc
import squidpy as sq
from anndata import AnnData

c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [53]:
# Create AnnData object
adata = AnnData(final_norm_matrix) #expression matrix transposed

adata.obsm["spatial"] = sample_metadata[['ROICoordinateX', 'ROICoordinateY']].to_numpy()  #roi coords
adata.obs["LEAPID"] = sample_metadata['LEAP_ID']
adata.obs["segment"] = sample_metadata['Segment']
adata.X = np.nan_to_num(adata.X, nan=0.0, posinf=0.0, neginf=0.0)

## Highly Variable Genes (HVGs)

In [54]:
from collections import defaultdict

# Dictionary to store HVG results per segment
hvg_results_by_segment = {}

# Loop over each segment type
for segment in ['CD45+','PanCK+']:
    print(f"Processing segment: {segment}")
    
    # Subset AnnData
    adata_sub = adata[adata.obs["segment"] == segment].copy()
    
    # Filter out genes with all-zero expression (optional but good)
    sc.pp.filter_genes(adata_sub, min_cells=1)
    
    # Identify highly variable genes
    sc.pp.highly_variable_genes(
        adata_sub,
        flavor='cell_ranger',
        n_top_genes=100
    )
    
    # Store DataFrame of HVGs
    hvg_df = adata_sub.var[adata_sub.var["highly_variable"]].copy()
    hvg_results_by_segment[segment] = hvg_df

    print(f"Found {hvg_df.shape[0]} highly variable genes for {segment}")


Processing segment: CD45+
Found 100 highly variable genes for CD45+
Processing segment: PanCK+
Found 100 highly variable genes for PanCK+


Get top 50 HVGs per segment

In [55]:
HVGs_cd45 = list(hvg_results_by_segment['CD45+'].sort_values(by='dispersions_norm', ascending=False)[:50].index)
HVGs_cd45

['KRT14',
 'FDCSP',
 'MUCL1',
 'IGLL5',
 'APOD',
 'IGHG1',
 'IGHA1',
 'S100A7',
 'IGHG4',
 'ZFP57',
 'IGHG2',
 'GABRP',
 'IGHM',
 'IGKC',
 'KRT23',
 'IGHG3',
 'MGP',
 'ACTG2',
 'FABP7',
 'CRABP1',
 'S100A1',
 'S100A8',
 'LCN2',
 'CRYAB',
 'ELN',
 'MMP7',
 'KRT17',
 'ELF5',
 'SERPINA3',
 'TRPM3',
 'TPSAB1',
 'CPB1',
 'KRT5',
 'MZT2B',
 'KRT15',
 'KRT6A',
 'AARD',
 'FNTB',
 'FXYD3',
 'C1QL2',
 'CCL24',
 'HLA-DQA1',
 'SFRP1',
 'SMR3A',
 'PPP2R2B',
 'JCHAIN',
 'SPP1',
 'CXCL14',
 'AREG',
 'CALML5']

For tumour (PanCK+) segments we need to filter out any immune genes from plasma cells that have contaminated the segment, otherwise these will dominate the analyses

In [56]:
hvg_results_by_segment['PanCK+'].index

Index(['CP', 'CYP1B1', 'KRT17', 'KRT14', 'IGF2', 'NCCRP1', 'C4B', 'TRPM3',
       'AARD', 'PEG10', 'KLK5', 'ELN', 'SERPINA3', 'ZFP57', 'CHI3L1', 'EN1',
       'FABP7', 'ACTG2', 'APOD', 'CFB', 'C1S', 'CPB1', 'CRYAB', 'DSG1', 'FNTB',
       'HLA-DQA1', 'HLA-DQB1', 'KRT81', 'LTF', 'SCGB2A2', 'MMP7', 'PI3',
       'SERPINB5', 'KLK6', 'S100A7', 'S100A9', 'CCL24', 'TFF3', 'IRS4',
       'EFEMP1', 'LBP', 'UCHL1', 'FOXA1', 'CXCL14', 'AQP3', 'KLK7', 'CALML3',
       'CSN3', 'SELENOP', 'IFI27', 'KRT6A', 'KRT16', 'LCN2', 'S100A2', 'WFDC2',
       'S100A1', 'CLCA2', 'CASP14', 'SPON2', 'PCSK1N', 'MSLN', 'GABRP',
       'KRT23', 'CALML5', 'ANK1', 'TTYH1', 'IRX1', 'MZT2B', 'ANKRD30A',
       'CCND1', 'KLK8', 'CCL28', 'FDCSP', 'TUBB2B', 'PPP2R2B', 'C1QL2',
       'STAC2', 'RARRES1', 'FN1', 'NDRG2', 'SPIB', 'AGR2', 'AZGP1', 'CA9',
       'CLU', 'CRABP1', 'CXCL1', 'CXCL8', 'KRT15', 'KRT5', 'MGP', 'MUCL1',
       'PIGR', 'S100A8', 'S100P', 'SOX10', 'SYT8', 'TRIM29', 'UBD', 'IDO1'],
      dtype='object')

Check none of the contamination genes made it into the HVGs

In [57]:
hvg_results_by_segment['PanCK+'].index.isin(contamination_genes).any()

False

Check whether the borderline plasma-correlated genes are in our list of HVGs

In [58]:
poss_genes = ['CD79A','TRBC1','C1QA','C1QB','C1QC','ADA2','LYZ','IL10RA','EVI2B','CORO1A','WIPF1']
hvg_results_by_segment['PanCK+'].index.isin(poss_genes).any()

False

In [59]:
HVGs_panck = list(hvg_results_by_segment['PanCK+'].sort_values(by='dispersions_norm', ascending=False)[:50].index)
HVGs_panck

['FDCSP',
 'MUCL1',
 'KRT14',
 'APOD',
 'S100A7',
 'S100A8',
 'SCGB2A2',
 'MGP',
 'ACTG2',
 'AQP3',
 'FABP7',
 'S100A9',
 'S100P',
 'CPB1',
 'KRT16',
 'ELN',
 'LCN2',
 'S100A2',
 'CRABP1',
 'CSN3',
 'KRT6A',
 'MMP7',
 'TFF3',
 'HLA-DQA1',
 'UBD',
 'AARD',
 'IRS4',
 'CXCL1',
 'RARRES1',
 'CASP14',
 'IRX1',
 'KRT17',
 'KLK5',
 'PIGR',
 'CCL28',
 'KRT5',
 'ANKRD30A',
 'CHI3L1',
 'GABRP',
 'FNTB',
 'CXCL8',
 'CA9',
 'FOXA1',
 'C4B',
 'LTF',
 'IGF2',
 'STAC2',
 'KRT81',
 'MSLN',
 'PI3']

In [60]:
from collections import defaultdict

top_n = 100
segment_to_top_genes = {
    segment: set(hvg_df.sort_values("dispersions_norm", ascending=False).index[:top_n])
    for segment, hvg_df in hvg_results_by_segment.items()
}

# Count how many segments each gene appears in
from collections import Counter

gene_counter = Counter()
for genes in segment_to_top_genes.values():
    gene_counter.update(genes)

# Show most common HVGs
print("Most frequently variable genes across segments:")
for gene, count in gene_counter.most_common(10):
    print(f"{gene}: {count} segments")

Most frequently variable genes across segments:
S100A7: 2 segments
EN1: 2 segments
LCN2: 2 segments
PCSK1N: 2 segments
CALML5: 2 segments
KRT15: 2 segments
PEG10: 2 segments
ZFP57: 2 segments
HLA-DQA1: 2 segments
KRT17: 2 segments


use the log transformed raw expression for the ground truth once we know what the HVGs are

In [61]:
hvgs_df_panck = log_transformed_expr.loc[rois_panck_train,HVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
hvgs_df_cd45 = log_transformed_expr.loc[rois_cd45_train,HVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [62]:

###hvgs_df_panck = expression_matrix.T.loc[rois_panck,HVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
###hvgs_df_cd45 = expression_matrix.T.loc[rois_cd45,HVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [63]:
hvgs_df_panck.to_csv(os.path.join(outcome_path,f"hvg_panck_raw.csv"), index=False)
hvgs_df_cd45.to_csv(os.path.join(outcome_path,f"hvg_cd45_raw.csv"), index=False)

TEST SET

In [64]:
hvgs_df_panck = log_transformed_expr.loc[rois_panck_test,HVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
hvgs_df_cd45 = log_transformed_expr.loc[rois_cd45_test,HVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})
hvgs_df_panck.to_csv(os.path.join(outcome_path,f"hvg_panck_raw_TEST.csv"), index=False)
hvgs_df_cd45.to_csv(os.path.join(outcome_path,f"hvg_cd45_raw_TEST.csv"), index=False)

## Spatially Variable Genes (SVGs)

In [65]:
results = {}

for leapid in adata.obs["LEAPID"].unique():
    for segment in ['CD45+','PanCK+']:
        subset = adata[
            (adata.obs["LEAPID"] == leapid) &
            (adata.obs["segment"] == segment)
        ].copy()
        

        # Compute spatial graph and spatial autocorrelation
        n_obs = subset.n_obs

        if n_obs > 1:
            #print(f"{leapid} {segment} {n_obs}")
            n_neighbors = min(5, n_obs - 1)


            # Build spatial graph and compute Moran’s I
            sq.gr.spatial_neighbors(subset, coord_type="generic", n_neighs=n_neighbors)
            sq.gr.spatial_autocorr(subset, mode="moran")

            # Extract top genes
            moranI = subset.uns['moranI']
            top_genes = moranI.sort_values("I", ascending=False).head(10)

            results[(leapid, segment)] = top_genes
            #print(f"Top genes for {leapid}-{segment}: {top_genes.index.tolist()}")

c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\squidpy\gr\_ppatterns.py:553: RuntimeWarning: invalid value encountered in divide
  z_norm = (score - params["expected"]) / seScore_norm
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\squidpy\gr\_ppatterns.py:553: RuntimeWarning: invalid value encountered i

In [66]:
from collections import defaultdict, Counter

# Dictionary: segment → Counter of gene frequencies
gene_counts_by_segment = defaultdict(Counter)

top_n = 20  # Adjust based on how many top genes you consider

for (leapid, segment), df in results.items():
    top_genes = df.head(top_n).index.tolist()
    gene_counts_by_segment[segment].update(top_genes)

Get top 50 SVGs per segment

In [67]:
SVGs_cd45 = [gene for segment, counter in gene_counts_by_segment.items() for gene,_ in counter.most_common(50) if segment=='CD45+']
SVGs_cd45

['SUPT7L',
 'RPL26',
 'COL1A1',
 'CRABP2',
 'IFI6',
 'PITX1',
 'RPLP1',
 'DMRTC1B',
 'PRMT8',
 'IGHG4',
 'IGHG3',
 'RPL41',
 'TMEM106C',
 'MX1',
 'S100A9',
 'ADAM15',
 'MMP11',
 'USP17L3',
 'SPP1',
 'KRT7',
 'MMP1',
 'BOD1L2',
 'EYA2',
 'C7',
 'IGHG2',
 'IGHG1',
 'CD82',
 'IGKC',
 'NFKB2',
 'P2RY12',
 'CCSAP',
 'IGHA1',
 'APOD',
 'UBE2E1',
 'PIP',
 'IFITM1',
 'ISG15',
 'GPR135',
 'APOC1',
 'TARP',
 'TAPBP',
 'S100A8',
 'TENT5C',
 'RPS20',
 'HLA-B',
 'REPS2',
 'ST6GALNAC3',
 'HLA-E',
 'APOE',
 'NDOR1']

In [68]:
SVGs_panck = [gene for segment, counter in gene_counts_by_segment.items() for gene,_ in counter.most_common(50) if segment=='PanCK+']
SVGs_panck

['SUPT7L',
 'HLA-B',
 'TMEM106C',
 'PTBP1',
 'ADAM15',
 'IFI27',
 'RPLP1',
 'CXCL14',
 'ATP8A1',
 'LTF',
 'FTH1',
 'ATP5MC2',
 'ALDOA',
 'FXYD3',
 'TPI1',
 'RPL13',
 'HLA-C',
 'HLA-F',
 'HLA-A',
 'RPL18',
 'H1-0',
 'PRMT8',
 'GABRP',
 'HSPA1B',
 'BRINP3',
 'STRN',
 'TMA7',
 'KRT81',
 'ENO1',
 'MIF',
 'CFB',
 'DMRTB1',
 'MUC1',
 'FABP5',
 'TAF6',
 'ISG15',
 'RNF126',
 'MGP',
 'CTSD',
 'KRT14',
 'SLC2A4RG',
 'S100P',
 'ANXA8',
 'H3C1',
 'C11orf58',
 'BLNK',
 'NWD2',
 'RPS12',
 'RPL12',
 'CMPK1']

Check whether any of the contamination genes are in the SVGs

In [69]:
set(SVGs_panck).intersection(set(contamination_genes))

set()

In [70]:
poss_genes = ['CD79A','TRBC1','C1QA','C1QB','C1QC','ADA2','LYZ','IL10RA','EVI2B','CORO1A','WIPF1']
set(SVGs_panck).intersection(set(poss_genes))

set()

In [71]:
svgs_df_panck = log_transformed_expr.loc[rois_panck_train,SVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
svgs_df_cd45 = log_transformed_expr.loc[rois_cd45_train,SVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [72]:

####svgs_df_panck = expression_matrix.T.loc[rois_panck,SVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
####svgs_df_cd45 = expression_matrix.T.loc[rois_cd45,SVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [73]:
svgs_df_panck.to_csv(os.path.join(outcome_path,f"svg_panck_raw.csv"), index=False)
svgs_df_cd45.to_csv(os.path.join(outcome_path,f"svg_cd45_raw.csv"), index=False)

TEST SET

In [74]:
svgs_df_panck = log_transformed_expr.loc[rois_panck_test,SVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
svgs_df_cd45 = log_transformed_expr.loc[rois_cd45_test,SVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})
svgs_df_panck.to_csv(os.path.join(outcome_path,f"svg_panck_raw_TEST.csv"), index=False)
svgs_df_cd45.to_csv(os.path.join(outcome_path,f"svg_cd45_raw_TEST.csv"), index=False)

Check overlap between SVGs and HVGs

In [75]:
set(SVGs_panck).intersection(set(HVGs_panck))

{'GABRP', 'KRT14', 'KRT81', 'LTF', 'MGP', 'S100P'}

In [76]:
set(SVGs_cd45).intersection(set(HVGs_cd45))

{'APOD', 'IGHA1', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGKC', 'S100A8', 'SPP1'}

## Gene Panels

In [77]:
gene_panel_cd45 = ['CD274','CXCL9','IFNG','BCL2','CD3D', 'CD8A', 'CD4', 'FOXP3', 'CD68', 'CD163','IDO1', 'LAG3', 'STAT1', 'S100A9']
gene_panel_panck = ['BRCA1','BRCA2', 'TP53', 'PIK3CA','MYC','EGFR', 'AR', 'PTEN', 'RB1', 'CD274','BCL2','KRT5', 'KRT14', 'TP63', 'MKI67', 'VIM', 'MMP9','SOX10', 'ALDH1A1', 'CD44', 'ZEB1', 'SNAI1', 'CCNB1']

panel_df_panck = log_transformed_expr.loc[rois_panck_train,gene_panel_panck].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_cd45 = log_transformed_expr.loc[rois_cd45_train,gene_panel_cd45].reset_index().rename(columns={"index": "ROI_ID"})

####panel_df_panck = expression_matrix.T.loc[rois_panck,gene_panel_panck].reset_index().rename(columns={"index": "ROI_ID"})
####panel_df_cd45 = expression_matrix.T.loc[rois_cd45,gene_panel_cd45].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_panck.to_csv(os.path.join(outcome_path,f"panel_panck_raw.csv"), index=False)
panel_df_cd45.to_csv(os.path.join(outcome_path,f"cd45_raw.csv"), index=False)

# TEST SET
panel_df_panck = log_transformed_expr.loc[rois_panck_test,gene_panel_panck].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_cd45 = log_transformed_expr.loc[rois_cd45_test,gene_panel_cd45].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_panck.to_csv(os.path.join(outcome_path,f"panel_panck_raw_TEST.csv"), index=False)
panel_df_cd45.to_csv(os.path.join(outcome_path,f"panel_cd45_raw_TEST.csv"), index=False)

# Check overlap of genes

In [155]:
df_panel = pd.DataFrame(data = panel_df_panck.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_panel['Geneset'] = 'Panel'
df_hvg = pd.DataFrame(data = hvgs_df_panck.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_hvg['Geneset'] = 'HVG'
df_svg = pd.DataFrame(data = svgs_df_panck.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_svg['Geneset'] = 'SVG'
df_panck = pd.concat([df_panel,df_hvg,df_svg]).reset_index(drop=True)
df_panck['Segment'] = 'PanCK+'
df_panck.shape

(123, 3)

In [145]:
duplicated_panck = list(df_panck[df_panck.duplicated('Gene')].Gene.unique())
print(f"{len(duplicated_panck)} genes are duplicated in panck")
print(duplicated_panck)
df_panck[df_panck['Gene'].isin(duplicated_panck)].sort_values(['Gene'])

7 genes are duplicated in panck
['KRT14', 'KRT5', 'LTF', 'GABRP', 'KRT81', 'MGP', 'S100P']


,Gene,Geneset,Segment
38,GABRP,HVG,PanCK+
22,GABRP,SVG,PanCK+
12,KRT14,Panel,PanCK+
2,KRT14,HVG,PanCK+
39,KRT14,SVG,PanCK+
11,KRT5,Panel,PanCK+
35,KRT5,HVG,PanCK+
47,KRT81,HVG,PanCK+
27,KRT81,SVG,PanCK+
44,LTF,HVG,PanCK+


In [171]:
unique_panck = list(df_panck['Gene'].unique())
print(f"PanCK+ unique genes: {len(unique_panck)}")

PanCK+ unique genes: 115


In [154]:
df_panel = pd.DataFrame(data = panel_df_cd45.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_panel['Geneset'] = 'Panel'
df_hvg = pd.DataFrame(data = hvgs_df_cd45.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_hvg['Geneset'] = 'HVG'
df_svg = pd.DataFrame(data = svgs_df_cd45.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_svg['Geneset'] = 'SVG'
df_cd45 = pd.concat([df_panel,df_hvg,df_svg]).reset_index(drop=True)
df_cd45['Segment'] = 'CD45+'
df_cd45.shape

(114, 3)

In [152]:
duplicated_cd45 = list(df_cd45[df_cd45.duplicated('Gene')].Gene.unique())
print(f"{len(duplicated_cd45)} genes are duplicated in panck")
print(duplicated_cd45)
df_cd45[df_cd45['Gene'].isin(duplicated_cd45)].sort_values(['Gene'])

10 genes are duplicated in panck
['IGHG4', 'IGHG3', 'S100A9', 'SPP1', 'IGHG2', 'IGHG1', 'IGKC', 'IGHA1', 'APOD', 'S100A8']


,Gene,Geneset,Segment
18,APOD,HVG,CD45+
96,APOD,SVG,CD45+
20,IGHA1,HVG,CD45+
95,IGHA1,SVG,CD45+
19,IGHG1,HVG,CD45+
89,IGHG1,SVG,CD45+
24,IGHG2,HVG,CD45+
88,IGHG2,SVG,CD45+
29,IGHG3,HVG,CD45+
74,IGHG3,SVG,CD45+


In [ ]:
unique_cd45 = list(df_cd45['Gene'].unique())


CD45+ unique genes: 104


In [194]:
print(f"CD45+ # unique genes: {len(unique_cd45)}")
print(f"PanCK+ # unique genes: {len(unique_panck)}")


CD45+ # unique genes: 104
PanCK+ # unique genes: 115


In [190]:
print(f"# genes across both segments: {len(set(unique_panck).union(set(unique_cd45)))}")

# genes across both segments: 186


In [191]:
print(f"# genes in both segments: {len(set(unique_panck).intersection(set(unique_cd45)))}")

# genes in both segments: 33


In [192]:
print(f"# genes only in CD45+: {len(set(unique_cd45).difference(set(unique_panck)))}")
print(f"# genes only in PanCK+: {len(set(unique_panck).difference(set(unique_cd45)))}")


# genes only in CD45+: 71
# genes only in PanCK+: 82


In [193]:
print(f"genes only in CD45+: {(set(unique_cd45).difference(set(unique_panck)))}")
print(f"genes only in PanCK+: {(set(unique_panck).difference(set(unique_cd45)))}")


genes only in CD45+: {'FOXP3', 'HLA-E', 'C1QL2', 'CRABP2', 'IFI6', 'CCL24', 'CCSAP', 'CD163', 'P2RY12', 'CRYAB', 'JCHAIN', 'CD68', 'IGHG4', 'RPL26', 'TENT5C', 'UBE2E1', 'IDO1', 'MMP1', 'EYA2', 'ELF5', 'MMP11', 'STAT1', 'IGLL5', 'CD8A', 'DMRTC1B', 'MX1', 'PITX1', 'APOC1', 'IFITM1', 'CALML5', 'NFKB2', 'S100A1', 'TPSAB1', 'SFRP1', 'KRT15', 'MZT2B', 'TAPBP', 'CD4', 'SERPINA3', 'IGHG2', 'CD3D', 'ST6GALNAC3', 'RPS20', 'ZFP57', 'CD82', 'LAG3', 'TARP', 'RPL41', 'IFNG', 'SMR3A', 'GPR135', 'TRPM3', 'IGHA1', 'C7', 'AREG', 'IGHG1', 'KRT23', 'IGHM', 'BOD1L2', 'USP17L3', 'NDOR1', 'REPS2', 'SPP1', 'COL1A1', 'APOE', 'KRT7', 'IGKC', 'PIP', 'PPP2R2B', 'IGHG3', 'CXCL9'}
genes only in PanCK+: {'ALDH1A1', 'DMRTB1', 'RPL13', 'NWD2', 'SOX10', 'PIGR', 'CHI3L1', 'TAF6', 'BRCA2', 'PIK3CA', 'UBD', 'ALDOA', 'FTH1', 'FOXA1', 'FABP5', 'CASP14', 'IRS4', 'ATP5MC2', 'AR', 'EGFR', 'BLNK', 'ENO1', 'PTEN', 'H3C1', 'H1-0', 'HLA-A', 'SNAI1', 'SCGB2A2', 'STRN', 'KLK5', 'PTBP1', 'ANXA8', 'LTF', 'BRINP3', 'HLA-F', 'IFI27', 'V

# Save Standard Deviation
useful for calculating nRMSE

In [78]:
print(len(set(gene_panel_panck + SVGs_panck )))
print(len(gene_panel_panck) + len(SVGs_panck))
print(set(gene_panel_panck).intersection(set(SVGs_panck)))

72
73
{'KRT14'}


In [79]:
allgenes_panck = list(set(gene_panel_panck + HVGs_panck + SVGs_panck))
allgenes_cd45 = list(set(gene_panel_cd45 + HVGs_cd45 + SVGs_cd45))

In [80]:
std_panck_test = pd.DataFrame(log_transformed_expr.loc[rois_panck_test,allgenes_panck].std(axis=0),columns=['std'])
std_cd45_test = pd.DataFrame(log_transformed_expr.loc[rois_cd45_test,allgenes_cd45].std(axis=0),columns=['std'])
std_panck_test.index.name='Gene'
std_cd45_test.index.name='Gene'
std_panck_test.to_csv(os.path.join(outcome_path,f"allgenes_panck_test_std.csv"), index=True)
std_cd45_test.to_csv(os.path.join(outcome_path,f"allgenes_cd45_test_std.csv"), index=True)

# Additional

I want to know which leap ids have high variance on the genes of interest

CD45:
Panel ['CXCL9', 'CD68', 'STAT1', 'S100A9']
['TENT5C',
 'CCSAP',
 'ST6GALNAC3',
 'UBE2E1',
 'CD82',
 'DMRTC1B']
PanCK:
Panel ['PIK3CA', 'EGFR', 'MKI67']
['MKI67',
 'PIK3CA',
 'CA9',
 'LCN2',
 'AQP3',
 'IGHM']

 genes_of_interest[('CD45+','Panel')] = ['BCL2', 'IDO1', 'STAT1', 'CD274', 'CD3D', 'CXCL9'] #Panel
genes_of_interest[('CD45+','HVG')] = ['AREG','FDCSP','CPB1']
genes_of_interest[('CD45+','SVG')] = ['ST6GALNAC3','PRMT8','C7','TARP', 'TENT5C','RPL41','P2RY12','PITX1','CCSAP' ] #/SVG
genes_of_interest[('PanCK+','Panel')] = ['BRCA1', 'SNAI1', 'PIK3CA', 'BRCA2', 'EGFR', 'SNAI1','PIK3CA', 'BRCA2','EGFR'] #Panel
genes_of_interest[('PanCK+','HVG')] = ['IGHG1', 'IGHG4', 'IGHG3', 'IGKC',  'IGHG2',   'CSN3', 'CA9', 'CPB1' ]
genes_of_interest[('PanCK+','SVG')] = ['ATP5MC2', 'BRINP3', 'DMRTB1', 'NWD2', 'ENO1']

In [81]:
cd45_genes_of_interest = ['CXCL9', 'CD68', 'STAT1', 'S100A9','TENT5C','CCSAP', 'ST6GALNAC3', 'UBE2E1', 'CD82', 'DMRTC1B']
panck_genes_of_interest = ['PIK3CA', 'EGFR', 'MKI67', 'CA9', 'LCN2', 'AQP3', 'IGHM']

In [82]:
cd45_genes_of_interest = ['BCL2', 'IDO1', 'STAT1', 'CD274','AREG','FDCSP','CPB1','ST6GALNAC3','PRMT8','C7','TARP']
panck_genes_of_interest = ['BRCA1', 'SNAI1', 'PIK3CA', 'IGHG1', 'IGHG4', 'IGKC','ATP5MC2', 'BRINP3', 'DMRTB1','BRCA2', 'EGFR']

In [83]:
expr_of_interest_panck = log_transformed_expr.loc[rois_panck_test,panck_genes_of_interest]
expr_of_interest_cd45 = log_transformed_expr.loc[rois_cd45_test,cd45_genes_of_interest]

In [84]:
expr_of_interest_cd45 = sample_metadata.loc[rois_cd45_test,['LEAP_ID']].join(expr_of_interest_cd45,how='left')
expr_of_interest_panck = sample_metadata.loc[rois_panck_test,['LEAP_ID']].join(expr_of_interest_panck,how='left')


In [85]:
for gene in cd45_genes_of_interest:
    print(f"{gene}     \t{list(expr_of_interest_cd45.groupby('LEAP_ID').std().sort_values(gene, ascending=False).index[:5])}")

BCL2     	['LEAP120', 'LEAP079', 'LEAP121', 'LEAP078', 'LEAP094']
IDO1     	['LEAP120', 'LEAP079', 'LEAP121', 'LEAP082', 'LEAP078']
STAT1     	['LEAP121', 'LEAP120', 'LEAP079', 'LEAP078', 'LEAP021']
CD274     	['LEAP079', 'LEAP120', 'LEAP121', 'LEAP078', 'LEAP082']
AREG     	['LEAP120', 'LEAP121', 'LEAP079', 'LEAP078', 'LEAP094']
FDCSP     	['LEAP078', 'LEAP082', 'LEAP120', 'LEAP079', 'LEAP121']
CPB1     	['LEAP079', 'LEAP120', 'LEAP078', 'LEAP045', 'LEAP121']
ST6GALNAC3     	['LEAP079', 'LEAP120', 'LEAP078', 'LEAP094', 'LEAP093']
PRMT8     	['LEAP078', 'LEAP121', 'LEAP120', 'LEAP079', 'LEAP094']
C7     	['LEAP120', 'LEAP079', 'LEAP121', 'LEAP078', 'LEAP021']
TARP     	['LEAP079', 'LEAP120', 'LEAP078', 'LEAP121', 'LEAP094']


In [86]:
for gene in panck_genes_of_interest:
    print(f"{gene}     \t{list(expr_of_interest_panck.groupby('LEAP_ID').std().sort_values(gene, ascending=False).index[:5])}")

BRCA1     	['LEAP065', 'LEAP121', 'LEAP082', 'LEAP120', 'LEAP093']
SNAI1     	['LEAP065', 'LEAP121', 'LEAP120', 'LEAP093', 'LEAP082']
PIK3CA     	['LEAP065', 'LEAP082', 'LEAP094', 'LEAP121', 'LEAP120']
IGHG1     	['LEAP094', 'LEAP082', 'LEAP121', 'LEAP065', 'LEAP064']
IGHG4     	['LEAP094', 'LEAP082', 'LEAP121', 'LEAP065', 'LEAP064']
IGKC     	['LEAP094', 'LEAP082', 'LEAP065', 'LEAP064', 'LEAP120']
ATP5MC2     	['LEAP065', 'LEAP082', 'LEAP120', 'LEAP064', 'LEAP094']
BRINP3     	['LEAP065', 'LEAP094', 'LEAP082', 'LEAP120', 'LEAP064']
DMRTB1     	['LEAP065', 'LEAP121', 'LEAP120', 'LEAP079', 'LEAP094']
BRCA2     	['LEAP065', 'LEAP121', 'LEAP094', 'LEAP120', 'LEAP082']
EGFR     	['LEAP120', 'LEAP065', 'LEAP121', 'LEAP094', 'LEAP082']


best for CD45
LEAP120
LEAP079
LEAP121

best for PanCK
LEAP121
LEAP065
LEAP082
LEAP120

In [87]:
log_transformed_expr.loc[rois_cd45_test,allgenes_cd45]

,USP17L3,BCL2,IFITM1,S100A7,TAPBP,IFNG,PPP2R2B,CD163,DMRTC1B,MZT2B,...,NFKB2,HLA-B,RPLP1,CD82,SMR3A,MUCL1,KRT6A,MMP7,IGHM,CCL24
DSP-1001660018794-B-E02.dcc,0.693147,0.693147,2.397895,0.693147,2.302585,0.693147,1.609438,2.564949,1.098612,2.302585,...,2.639057,5.030438,3.871201,1.098612,1.945910,2.708050,4.477337,0.693147,2.484907,2.079442
DSP-1001660020771-B-C09.dcc,4.844187,4.595120,5.680173,3.784190,5.332719,3.637586,5.370638,4.553877,3.761200,7.709308,...,6.306275,8.468213,7.261927,4.672829,2.772589,3.367296,3.637586,4.043051,4.418841,3.367296
DSP-1001660016632-A-D05.dcc,0.693147,0.693147,1.098612,0.693147,0.693147,0.693147,1.098612,0.693147,0.693147,1.098612,...,0.693147,1.791759,2.302585,1.098612,1.386294,0.693147,0.693147,0.693147,0.693147,1.098612
DSP-1001660018904-B-C02.dcc,3.891820,3.178054,5.393628,2.397895,3.951244,2.302585,2.079442,3.465736,2.772589,3.970292,...,3.871201,6.658011,6.049733,3.258097,2.302585,2.302585,2.397895,3.135494,5.509388,2.397895
DSP-1001660018904-B-E07.dcc,3.637586,2.302585,3.988984,1.791759,3.091042,1.945910,1.945910,1.945910,2.197225,2.995732,...,3.850148,5.613128,4.584967,2.564949,1.386294,2.079442,1.791759,2.197225,5.023881,2.397895
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DSP-1001660018904-B-D02.dcc,4.718499,4.110874,6.222576,3.663562,4.882802,3.663562,2.995732,4.564348,3.496508,4.770685,...,4.859812,7.475906,7.137278,4.317488,3.891820,2.890372,3.713572,3.465736,4.912655,3.583519
DSP-1001660020771-B-D01.dcc,1.386294,1.098612,2.484907,0.693147,1.791759,0.693147,1.791759,0.693147,0.693147,4.276666,...,2.079442,4.762174,2.564949,1.386294,0.693147,1.098612,0.693147,1.098612,0.693147,0.693147
DSP-1001660014498-F-B05.dcc,2.302585,2.708050,4.077537,4.941642,3.044522,2.397895,4.007333,3.496508,2.397895,4.025352,...,5.214936,5.831882,4.804021,2.890372,2.772589,2.772589,3.526361,2.944439,2.708050,4.442651
DSP-1001660020485-C-D02.dcc,3.178054,1.791759,2.197225,3.091042,2.484907,1.386294,2.397895,2.079442,1.098612,4.564348,...,3.496508,5.204007,3.970292,2.302585,1.609438,2.302585,2.639057,1.098612,1.098612,1.791759


# END
